In [2]:
# =========================================================
# SECTION III-A: Manifest construction & subject-level splitting
# =========================================================
# This replaces the old placeholder that hardcoded age=73.0 for every scan.
# Real ages + CDR status are pulled from the OASIS-3 metadata CSVs and
# joined onto every preprocessed scan by parsing its filename.

import re
import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


def parse_preproc_filename(path: str) -> dict:
    """
    Parses subject/session/run info out of filenames produced by
    run_preprocessing(), e.g.:
        sub-OAS30001_sess-d0129_run-02_T1w_preproc.npz
    and reconstructs the 'MRI Label' key used by the OASIS-3 metadata CSVs
    (e.g. 'OAS30001_MR_d0129') so scans can be joined to age/CDR.
    """
    fname = os.path.basename(path)
    m = re.search(r"sub-(OAS\d+)_sess-d(\d+)_run-(\d+)", fname)
    if not m:
        raise ValueError(f"Could not parse subject/session/run from filename: {fname}")
    subject_id, day_str, run = m.group(1), m.group(2), m.group(3)
    mri_label = f"{subject_id}_MR_d{day_str}"
    return {
        "subject_id": subject_id,
        "day": int(day_str),
        "run": run,
        "mri_label": mri_label,
        "file_path": path,
    }


def build_manifest(preproc_dir: str = "./preprocessed",
                    scan_meta_csv: str = "okarun143_8_17_2026_12_36_37.csv",
                    cdr_csv: str = "OASIS3_UDSb4_cdr_with_mri_fixed.csv") -> pd.DataFrame:
    """
    Builds one row per preprocessed scan with real age + CDR status:
      subject_id, day, run, mri_label, file_path, age, sex, CDRTOT, CDRSUM, MMSE
    """
    files = sorted(glob.glob(os.path.join(preproc_dir, "*_preproc.npz")))
    if not files:
        raise FileNotFoundError(
            f"No preprocessed .npz files found in {preproc_dir}. "
            f"Run run_preprocessing() or batch_preprocess() first."
        )

    manifest = pd.DataFrame([parse_preproc_filename(f) for f in files])

    scan_meta = (
        pd.read_csv(scan_meta_csv)[["Label", "Age", "M/F"]]
        .rename(columns={"Label": "mri_label", "Age": "age_meta", "M/F": "sex"})
    )
    cdr = (
        pd.read_csv(cdr_csv)[["MRI Label", "age at visit", "CDRTOT", "CDRSUM", "MMSE"]]
        .rename(columns={"MRI Label": "mri_label", "age at visit": "age_cdr"})
    )

    manifest = manifest.merge(scan_meta, on="mri_label", how="left")
    manifest = manifest.merge(cdr, on="mri_label", how="left")

    # Prefer the CDR visit's recorded age (matches the clinical timeline used
    # for diagnosis); fall back to the imaging-session age if no CDR row exists.
    manifest["age"] = manifest["age_cdr"].fillna(manifest["age_meta"])
    manifest = manifest.drop(columns=["age_meta", "age_cdr"])

    missing = manifest[manifest["age"].isna()]
    if len(missing):
        print(f"Warning: dropping {len(missing)} scan(s) with no matching metadata row:")
        print(missing["file_path"].tolist())
    manifest = manifest.dropna(subset=["age"]).reset_index(drop=True)

    return manifest


def subject_level_split(manifest: pd.DataFrame,
                         test_size: float = 0.15,
                         val_size: float = 0.15,
                         random_state: int = 42):
    """
    III-A: every scan/session/repeat belonging to a subject stays in exactly
    one fold. The split is performed on unique subject_id, never on rows,
    so no subject's anatomy can leak between train/val/test.
    """
    subjects = manifest["subject_id"].unique()
    if len(subjects) < 3:
        return manifest.copy(), manifest.copy(), manifest.copy()
        
    train_subj, test_subj = train_test_split(
        subjects, test_size=test_size, random_state=random_state
    )
    train_subj, val_subj = train_test_split(
        train_subj, test_size=val_size / (1 - test_size), random_state=random_state
    )

    train_df = manifest[manifest["subject_id"].isin(train_subj)].reset_index(drop=True)
    val_df = manifest[manifest["subject_id"].isin(val_subj)].reset_index(drop=True)
    test_df = manifest[manifest["subject_id"].isin(test_subj)].reset_index(drop=True)

    assert not (set(train_df["subject_id"]) & set(val_df["subject_id"]))
    assert not (set(train_df["subject_id"]) & set(test_df["subject_id"]))
    assert not (set(val_df["subject_id"]) & set(test_df["subject_id"]))

    return train_df, val_df, test_df


# ## Section III-B -- Age-bias correction

In [3]:
# =========================================================
# SECTION III-B: Systemic age-bias correction (Eq. 3-4)
# =========================================================
from scipy.stats import linregress


def fit_bias_correction(train_predictions_cn: pd.DataFrame) -> dict:
    """
    Fits yhat = alpha*y + beta (Eq. 3) on TRAINING-SET, CDR 0.0 subjects only.
    train_predictions_cn must have columns: 'age', 'predicted_age'.
    NEVER re-fit this on validation/test data or on impaired subjects --
    doing so would erase the disease-driven atrophy signal (Sec III-B).
    """
    alpha, beta, r_value, p_value, se = linregress(
        train_predictions_cn["age"], train_predictions_cn["predicted_age"]
    )
    print(f"Bias correction fit on n={len(train_predictions_cn)} CN scans: "
          f"alpha={alpha:.4f}, beta={beta:.4f}, r={r_value:.3f}")
    return {"alpha": alpha, "beta": beta}


def apply_bias_correction(df: pd.DataFrame, params: dict) -> pd.DataFrame:
    """
    Applies the stored (alpha, beta) to ALL scans (healthy + impaired) and
    computes the Brain Age Gap: BAG = yhat_bc - y (Eq. 4).
    """
    df = df.copy()
    df["predicted_age_bc"] = (df["predicted_age"] - params["beta"]) / params["alpha"]
    df["BAG"] = df["predicted_age_bc"] - df["age"]
    return df


# ## Section III-C -- Ensemble training + inference (fixes the age=73.0 bug)

In [4]:
# =========================================================
# SECTION III-C: Ensemble training + inference driver
# (this is what actually fixes the hardcoded age=73.0 problem end-to-end)
# =========================================================
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np


def train_single_model(train_df, val_df, seed, device, epochs=30, lr=1e-4, batch_size=4):
    """
    Trains one SFCN ensemble member on CDR-0.0 TRAIN scans only (III-C.2),
    early-stopping on the CDR-0.0 held-out VAL set.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = SFCN().to(device)
    criterion = nn.L1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_loader = DataLoader(BrainDataset(train_df), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(BrainDataset(val_df), batch_size=batch_size, shuffle=False)

    best_val_mae = float("inf")
    best_state = None
    patience, patience_ctr = 5, 0

    for epoch in range(epochs):
        train_loss, train_mae = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_mae = evaluate(model, val_loader, criterion, device)
        print(f"[seed {seed}] epoch {epoch+1}/{epochs} "
              f"train_mae={train_mae:.2f} val_mae={val_mae:.2f}")

        if val_mae < best_val_mae:
            best_val_mae, best_state, patience_ctr = val_mae, model.state_dict(), 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"[seed {seed}] early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(best_state)
    return model, best_val_mae


def train_ensemble(train_df, val_df, device, n_models=5, base_seed=0, **train_kwargs):
    """
    III-C.4: trains multiple SFCN instances differing only by random seed --
    a lighter-weight stand-in for also mixing in a second architecture such
    as 3D-ResNet (swap SFCN() above for another network to extend this).
    """
    models = []
    for i in range(n_models):
        model, val_mae = train_single_model(train_df, val_df, seed=base_seed + i,
                                             device=device, **train_kwargs)
        models.append(model)
    return models


@torch.no_grad()
def ensemble_predict(models, manifest_df, device, batch_size=4):
    """
    Runs every ensemble member over every scan in manifest_df and returns
    manifest_df with two new columns:
      predicted_age    -- ensemble mean (fed into BAG downstream)
      predicted_age_sd -- ensemble spread (per-scan uncertainty estimate,
                           to be checked against the real same-day noise in
                           Section III-E)
    """
    loader = DataLoader(BrainDataset(manifest_df), batch_size=batch_size, shuffle=False)

    all_model_preds = []
    for model in models:
        model.eval()
        preds = []
        for inputs, _ in loader:
            inputs = inputs.to(device)
            preds.append(model(inputs).cpu().numpy())
        all_model_preds.append(np.concatenate(preds))

    all_model_preds = np.stack(all_model_preds, axis=0)  # (n_models, n_scans)
    out = manifest_df.copy()
    out["predicted_age"] = all_model_preds.mean(axis=0)
    out["predicted_age_sd"] = all_model_preds.std(axis=0)
    return out


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # ---- III-A: real manifest + subject-level split (fixes age=73 bug) ----
    manifest = build_manifest()
    print(f"Loaded manifest: {len(manifest)} scans across "
          f"{manifest['subject_id'].nunique()} subjects.")

    train_df, val_df, test_df = subject_level_split(manifest)

    # ---- III-C.2: train + early-stop on cognitively normal (CDR 0.0) only --
    train_cn = train_df[train_df["CDRTOT"] == 0.0].reset_index(drop=True)
    val_cn = val_df[val_df["CDRTOT"] == 0.0].reset_index(drop=True)
    print(f"Training on {len(train_cn)} CDR-0.0 scans "
          f"({train_cn['subject_id'].nunique()} subjects); "
          f"validating on {len(val_cn)} CDR-0.0 scans.")

    if len(train_cn) == 0:
        raise RuntimeError(
            "No CDR 0.0 scans available to train on. Run batch_preprocess() "
            "over more of the cohort first -- a single preprocessed test "
            "subject is only enough to validate the preprocessing pipeline, "
            "not to train a model."
        )

    models = train_ensemble(train_cn, val_cn, device, n_models=5)

    # ---- III-B: bias correction fit on TRAIN CDR-0.0 predictions only -----
    train_cn_pred = ensemble_predict(models, train_cn, device)
    bias_params = fit_bias_correction(train_cn_pred)

    # ---- apply to every scan (healthy + impaired) --------------------------
    all_pred = ensemble_predict(models, manifest, device)
    all_pred = apply_bias_correction(all_pred, bias_params)

    all_pred.to_csv("brainage_predictions.csv", index=False)
    print("Saved per-scan predictions + BAG to brainage_predictions.csv")
    return all_pred, models, bias_params


if __name__ == "__main__":
    all_pred, models, bias_params = main()


# ## Section III-D -- Longitudinal trajectory tracking

Using device: cuda
Loaded manifest: 1 scans across 1 subjects.
Training on 1 CDR-0.0 scans (1 subjects); validating on 1 CDR-0.0 scans.


NameError: name 'SFCN' is not defined

In [ ]:
# =========================================================
# SECTION III-D: Longitudinal trajectory tracking (Eq. 6-9)
# =========================================================
from scipy.stats import linregress
import statsmodels.formula.api as smf


def add_followup_time(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds t_years = years elapsed since each subject's first scan (the t_i,j
    used in Eq. 6-7). Assumes 'day' = OASIS "days since entry" integer.
    """
    df = df.sort_values(["subject_id", "day"]).copy()
    baseline_day = df.groupby("subject_id")["day"].transform("min")
    df["t_years"] = (df["day"] - baseline_day) / 365.25
    return df


def per_subject_ols(long_df: pd.DataFrame, min_visits: int = 3) -> pd.DataFrame:
    """
    EXPLORATORY ONLY, per the paper: fits BAG = S_i * t + C_i per subject.
    Requires >= 3 visits (III-D.2). Not used for formal inference --
    see fit_lme_trajectories() below for the pooled model that is.
    """
    rows = []
    for subject_id, g in long_df.groupby("subject_id"):
        if len(g) < min_visits:
            continue
        slope, intercept, r, p, se = linregress(g["t_years"], g["BAG"])
        rows.append({
            "subject_id": subject_id,
            "S_raw_ols": slope,
            "C_ols": intercept,
            "n_visits": len(g),
            "follow_up_years": g["t_years"].max(),
        })
    return pd.DataFrame(rows)


def fit_lme_trajectories(long_df: pd.DataFrame, min_visits: int = 3):
    """
    Formal inference model: a linear mixed-effects model with a random
    slope and intercept per subject, stabilizing per-subject slopes via
    partial pooling (Eq. 7's caveat about noisy per-subject OLS).
    Returns (fitted statsmodels result, per-subject stabilized slope table).
    """
    eligible = long_df.groupby("subject_id").filter(lambda g: len(g) >= min_visits)
    model = smf.mixedlm(
        "BAG ~ t_years", data=eligible, groups=eligible["subject_id"],
        re_formula="~t_years"
    )
    result = model.fit()
    print(result.summary())

    slopes = {
        subj: re["t_years"] + result.fe_params["t_years"]
        for subj, re in result.random_effects.items()
    }
    S_raw_lme = pd.DataFrame({
        "subject_id": list(slopes.keys()),
        "S_raw": list(slopes.values()),
    })
    return result, S_raw_lme


# ## Section III-E -- Noise floor, cognitive stratification, smallest-detectable-slope

In [ ]:
# =========================================================
# SECTION III-E: Noise quantification & smallest-detectable-slope (Eq. 10-12)
# =========================================================
from scipy.stats import levene


def build_repeat_pairs(pred_df: pd.DataFrame) -> pd.DataFrame:
    """
    Finds same-day repeat T1w pairs (same mri_label, different 'run') and
    returns one row per pair with both scans' BAG -- the basis for the
    1,027-repeat-scan noise-floor analysis.
    """
    rows = []
    for mri_label, g in pred_df.groupby("mri_label"):
        g = g.sort_values("run")
        if len(g) < 2:
            continue
        r1, r2 = g.iloc[0], g.iloc[1]
        rows.append({
            "subject_id": r1["subject_id"],
            "mri_label": mri_label,
            "BAG_run1": r1["BAG"],
            "BAG_run2": r2["BAG"],
            "CDRTOT": r1["CDRTOT"],
        })
    return pd.DataFrame(rows)


def compute_noise_floor(repeats_df: pd.DataFrame):
    """
    Eq. 10-11: same-day repeats contain zero true biological aging, so any
    difference in BAG between the two runs is pure measurement error.
    """
    d = repeats_df["BAG_run1"] - repeats_df["BAG_run2"]
    sd_d = d.std(ddof=1)
    sigma_sem = sd_d / np.sqrt(2)          # Eq. 10
    mdc = 1.96 * np.sqrt(2) * sigma_sem    # Eq. 11
    return sigma_sem, mdc, d


def stratified_noise_floor(repeats_df: pd.DataFrame):
    """
    III-E.3: separate noise floors for CDR 0.0 vs CDR > 0.0 repeat pairs,
    compared with Levene's test for unequal variance.
    """
    normal = repeats_df[repeats_df["CDRTOT"] == 0.0]
    impaired = repeats_df[repeats_df["CDRTOT"] > 0.0]

    sigma_n, mdc_n, d_n = compute_noise_floor(normal)
    sigma_i, mdc_i, d_i = compute_noise_floor(impaired)
    stat, p = levene(d_n, d_i)

    print(f"CDR 0.0   (n={len(normal)}): sigma_SEM={sigma_n:.3f} yrs, MDC={mdc_n:.3f} yrs")
    print(f"CDR > 0.0 (n={len(impaired)}): sigma_SEM={sigma_i:.3f} yrs, MDC={mdc_i:.3f} yrs")
    print(f"Levene's test for unequal noise variance: stat={stat:.3f}, p={p:.4f}")

    return {
        "sigma_sem_normal": sigma_n, "mdc_normal": mdc_n,
        "sigma_sem_impaired": sigma_i, "mdc_impaired": mdc_i,
        "levene_stat": stat, "levene_p": p,
    }


def se_of_slope(sigma_noise: float, t_values: np.ndarray) -> float:
    """
    Eq. 8: standard error of an OLS slope estimated from measurements with
    per-timepoint noise SD sigma_noise (the sigma_SEM from Eq. 10) --
    the standard SE(slope) = sigma / sqrt(sum((t - t_bar)^2)) result.
    """
    t_bar = np.mean(t_values)
    ss_t = np.sum((t_values - t_bar) ** 2)
    return sigma_noise / np.sqrt(ss_t)


def clean_speed(S_raw: float, se_s: float) -> float:
    """Eq. 12: zero out slopes that don't clear the 1.96*SE noise threshold."""
    return S_raw if abs(S_raw) > 1.96 * se_s else 0.0


def compute_clean_speeds(trajectory_df: pd.DataFrame, long_df: pd.DataFrame,
                          sigma_noise: float) -> pd.DataFrame:
    """
    trajectory_df must have columns: subject_id, S_raw (from per_subject_ols
    or fit_lme_trajectories). long_df supplies each subject's t_years values
    to compute SE(S_i).
    """
    out = trajectory_df.copy()
    se_list, clean_list = [], []
    for _, row in out.iterrows():
        t_vals = long_df.loc[long_df["subject_id"] == row["subject_id"], "t_years"].values
        se_s = se_of_slope(sigma_noise, t_vals)
        se_list.append(se_s)
        clean_list.append(clean_speed(row["S_raw"], se_s))
    out["SE_S"] = se_list
    out["S_clean"] = clean_list
    return out


# ## Section III-F/G -- Clinical validation (Cox PH + C-index comparison)

In [ ]:
# =========================================================
# SECTION III-F/G: Clinical validation via Cox Proportional Hazards
# =========================================================
# pip install lifelines   (already added to the setup cell at the top)
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index


def build_survival_table(trajectory_df: pd.DataFrame,
                          diagnoses_df: pd.DataFrame,
                          baseline_window_visits: int = 3) -> pd.DataFrame:
    """
    III-F.2/G.2-3: the aging speed is estimated from each subject's first
    `baseline_window_visits` scans; the survival clock starts at the end of
    that window (prevents immortal-time bias). Converters = CDR 0.0
    throughout the baseline window, then CDR > 0.0 at a later visit.
    Non-converters = CDR 0.0 throughout all post-baseline visits.

    diagnoses_df: long-format table with subject_id, day, age, sex, CDRTOT
    per visit (built from OASIS3_UDSb4_cdr_final.csv).
    """
    rows = []
    for subject_id, g in diagnoses_df.sort_values("day").groupby("subject_id"):
        baseline = g.iloc[:baseline_window_visits]
        if len(baseline) < baseline_window_visits:
            continue
        if not (baseline["CDRTOT"] == 0.0).all():
            continue  # must be CDR 0.0 throughout the baseline window

        post = g.iloc[baseline_window_visits:]
        baseline_end_day = baseline["day"].max()

        converted = post[post["CDRTOT"] > 0.0]
        if len(converted):
            event_day = converted["day"].iloc[0]
            event_observed = 1
        elif len(post):
            event_day = post["day"].iloc[-1]
            event_observed = 0
        else:
            continue  # no post-baseline follow-up -- can't use for survival

        rows.append({
            "subject_id": subject_id,
            "time_to_event": (event_day - baseline_end_day) / 365.25,
            "event_observed": event_observed,
            "age": baseline["age"].iloc[0],
            "sex": baseline["sex"].iloc[0] if "sex" in baseline.columns else np.nan,
            "N": baseline_window_visits,
        })

    surv_df = pd.DataFrame(rows)
    surv_df = surv_df.merge(trajectory_df, on="subject_id", how="inner")
    surv_df["sex"] = surv_df["sex"].map({"M": 0, "F": 1}).fillna(surv_df["sex"])
    return surv_df.dropna()


def fit_cox_models(surv_df: pd.DataFrame):
    """
    Fits Model 1 (raw speed, Eq. 15) and Model 2 (noise-subtracted clean
    speed, Eq. 16), both controlling for baseline gap C, age, sex, and N.
    """
    cols_raw = ["time_to_event", "event_observed", "S_raw", "C", "age", "sex", "N"]
    cols_clean = ["time_to_event", "event_observed", "S_clean", "C", "age", "sex", "N"]

    cph_raw = CoxPHFitter()
    cph_raw.fit(surv_df[cols_raw], duration_col="time_to_event", event_col="event_observed")

    cph_clean = CoxPHFitter()
    cph_clean.fit(surv_df[cols_clean], duration_col="time_to_event", event_col="event_observed")

    c_raw = concordance_index(surv_df["time_to_event"],
                               -cph_raw.predict_partial_hazard(surv_df),
                               surv_df["event_observed"])
    c_clean = concordance_index(surv_df["time_to_event"],
                                 -cph_clean.predict_partial_hazard(surv_df),
                                 surv_df["event_observed"])

    print(f"Harrell's C-index -- raw speed model:   {c_raw:.4f}")
    print(f"Harrell's C-index -- clean speed model: {c_clean:.4f}")
    return cph_raw, cph_clean, c_raw, c_clean


def bootstrap_cindex_diff(surv_df: pd.DataFrame, cph_raw, cph_clean,
                           n_boot: int = 1000, random_state: int = 42):
    """
    III-G.5: paired bootstrap (1000 iterations) on the C-index difference
    (clean - raw), reporting the mean and 95% CI.
    """
    rng = np.random.default_rng(random_state)
    n = len(surv_df)
    diffs = []
    for _ in range(n_boot):
        sample = surv_df.iloc[rng.integers(0, n, n)]
        if sample["event_observed"].sum() == 0:
            continue
        try:
            c_r = concordance_index(sample["time_to_event"],
                                     -cph_raw.predict_partial_hazard(sample),
                                     sample["event_observed"])
            c_c = concordance_index(sample["time_to_event"],
                                     -cph_clean.predict_partial_hazard(sample),
                                     sample["event_observed"])
            diffs.append(c_c - c_r)
        except Exception:
            continue

    diffs = np.array(diffs)
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
    print(f"Bootstrap C-index diff (clean - raw): mean={diffs.mean():.4f}, "
          f"95% CI=[{ci_low:.4f}, {ci_high:.4f}]")
    return diffs.mean(), (ci_low, ci_high)